<a href="https://colab.research.google.com/github/MightyCrimsonX/Crimson-Notebooks/blob/main/Notebooks/MightyCrimson_PixaiTaggerGradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏷️ PixAI Tagger v1.0 Tagger (Por Mighty Crimson)

Versión adaptada para **Google Colab** del Space oficial [pixai-labs/pixai-tagger-demo](https://huggingface.co/spaces/pixai-labs/pixai-tagger-demo).

⚡ **Detección Automática de Hardware:**
- **En GPU (T4 gratuita en Colab):** Se activa automáticamente en **CUDA con FP16**
- **En CPU:** Se ejecuta optimizado en **FP32 nativo**, Tiempo estimado por imagen: **120 - 140 segundos**

### 📦 Celda 1: Instalación, Configuración de Hardware (CPU/GPU) y Carga del Modelo
Instala dependencias, detecta si dispones de GPU o CPU y carga el modelo `pixai-labs/pixai-tagger-v1.0`

In [ ]:
#@markdown # Instalacion y dependencias
# ==============================================================================
# 1. INSTALACIÓN DE DEPENDENCIAS Y CONFIGURACIÓN HÍBRIDA CPU / GPU
# ==============================================================================
!pip install  gradio transformers huggingface_hub timm torchvision pillow numpy

import os
import time
import logging
from pathlib import Path
from functools import lru_cache

import torch
from huggingface_hub import snapshot_download
from PIL import Image
from transformers import pipeline

# Detección inteligente de aceleración de hardware
HAS_CUDA = torch.cuda.is_available()
if HAS_CUDA:
    gpu_name = torch.cuda.get_device_name(0)
    DEVICE_ARG = 0
    DTYPE_ARG = torch.float16
    print(f"🚀 GPU DETECTADA: {gpu_name}")
    print("⚡ Modo: CUDA con precisión FP16 activada (Inferencia ultra-rápida ~1s).")
else:
    DEVICE_ARG = -1
    DTYPE_ARG = torch.float32
    num_threads = os.cpu_count() or 2
    torch.set_num_threads(num_threads)
    try:
        torch.set_num_interop_threads(1)
    except Exception:
        pass
    if hasattr(torch.backends, 'mkldnn'):
        torch.backends.mkldnn.enabled = True
    print(f"⚙️ GPU no detectada. Modo: CPU.")
    print("💡 Consejo: Si deseas inferencias en 1 segundo, ve a 'Entorno de ejecución > Cambiar tipo de entorno de ejecución' y elige 'T4 GPU'.")

# Categorías y constantes
CATEGORIES = ('general', 'character', 'clothing', 'pose', 'style', 'copyright', 'meta', 'rating')
BASE_CATEGORIES = ('general', 'character', 'style', 'copyright', 'meta', 'rating')

LABELS = {
    'general': 'General',
    'character': 'Character',
    'clothing': 'Clothing',
    'pose': 'Pose',
    'style': 'Style',
    'copyright': 'Copyright / series',
    'meta': 'Meta',
    'rating': 'Rating'
}

COMBINED_ORDER = ('character', 'copyright', 'clothing', 'pose', 'general', 'style', 'meta', 'rating')

MODEL_REPO_ID = 'pixai-labs/pixai-tagger-v1.0'
MODEL_REVISION = 'f33cfdb53c0c90b049bab9ce066eea1118970ef8'

# Reglas semánticas para filtrado de ropa y pose
CLOTHING_STEMS = {
    'shirt', 'blouse', 'top', 'sleeves', 'sleeveless', 'hoodie', 'sweater', 'cardigan',
    'vest', 'jacket', 'coat', 'blazer', 'turtleneck', 'corset', 'skirt', 'pants', 'shorts',
    'jeans', 'panties', 'boxers', 'bloomers', 'miniskirt', 'dress', 'sundress', 'gown',
    'robe', 'kimono', 'yukata', 'cheongsam', 'qipao', 'uniform', 'suit', 'tuxedo', 'costume',
    'leotard', 'swimsuit', 'bikini', 'monokini', 'apron', 'maid', 'armor', 'armour', 'pajamas',
    'cloak', 'cape', 'underwear', 'bra', 'lingerie', 'garter', 'socks', 'stockings',
    'thighhighs', 'kneehighs', 'pantyhose', 'tights', 'shoes', 'boots', 'sneakers', 'sandals',
    'slippers', 'heels', 'loafers', 'footwear', 'hat', 'cap', 'beret', 'beanie', 'hood',
    'bonnet', 'helmet', 'tiara', 'crown', 'headband', 'hairband', 'hair_ornament', 'hair_ribbon',
    'hair_bow', 'ribbon', 'bow', 'tie', 'necktie', 'bowtie', 'scarf', 'muffler', 'choker',
    'collar', 'belt', 'gloves', 'mittens', 'cuffs', 'wristband', 'mask', 'glasses', 'sunglasses',
    'eyepatch', 'jewelry', 'necklace', 'earrings', 'ring', 'bracelet', 'anklet', 'veil',
    'pleated', 'frills', 'ruffles', 'lace', 'tutu', 'sarong', 'bodysuit', 'sarashi',
    'collared', 'open_clothes', 'bare_shoulders', 'glove', 'sock', 'boot', 'shoe'
}

CLOTHING_EXCLUDE = {
    'collarbone', 'rainbow', 'elbow', 'bowing', 'arm_held_by_clothing', 'bow_weapon',
    'holding_bow', 'carrying_bow', 'bow_hair', 'eyebrow', 'eyebrows'
}

POSE_STEMS = {
    'standing', 'sitting', 'lying', 'kneeling', 'squatting', 'crouching', 'floating',
    'leaning', 'straddling', 'all_fours', 'on_back', 'on_stomach', 'on_side', 'fetal_position',
    'seiza', 'wariza', 'looking_at_viewer', 'looking_away', 'looking_back', 'looking_at_another',
    'looking_down', 'looking_up', 'looking_to_the_side', 'sideways_glance', 'head_tilt',
    'head_back', 'turning_around', 'profile', 'holding', 'arms_up', 'arms_down',
    'arms_behind_back', 'arms_behind_head', 'crossed_arms', 'hand_on_hip', 'hands_on_hips',
    'hand_to_mouth', 'hand_on_cheek', 'hand_on_chest', 'hands_in_pockets', 'reaching',
    'pointing', 'peace_sign', 'v_gesture', 'thumbs_up', 'claw_pose', 'waving', 'salute',
    'cat_pose', 'crossed_legs', 'spread_legs', 'legs_apart', 'legs_together', 'knees_up',
    'knees_together', 'legs_up', 'bent_knee', 'walking', 'running', 'jumping', 'falling',
    'dancing', 'fighting_stance', 'stretching', 'full_body', 'upper_body', 'cowboy_shot',
    'from_above', 'from_below', 'from_behind', 'from_side', 'dutch_angle', 'bent_over',
    'arms_at_sides', 'hands_on_head', 'hands_clasped', 'hand_on_another', 'hug', 'kiss',
    'carrying', 'riding', 'sitting_on', 'sleeping', 'winking'
}

POSE_EXCLUDE = {
    'holding_breath', 'looking_glasses'
}

def is_clothing_tag(tag: str) -> bool:
    if tag in CLOTHING_EXCLUDE:
        return False
    parts = tag.split('_')
    for stem in CLOTHING_STEMS:
        if stem in parts:
            return True
        if '_' in stem and stem in tag:
            return True
    return False

def is_pose_tag(tag: str) -> bool:
    if tag in POSE_EXCLUDE:
        return False
    for stem in POSE_STEMS:
        if stem == tag or stem in tag:
            return True
    return False


class EndpointHandler:
    def __init__(self, model_dir=None):
        local_dir = model_dir or os.environ.get('MODEL_DIR')
        if local_dir:
            source = Path(local_dir).resolve()
            if not (source / 'model.safetensors').is_file():
                raise RuntimeError('MODEL_DIR debe contener model.safetensors.')
        else:
            token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
            print("⏳ Descargando/verificando modelo PixAI Tagger v1.0 desde Hugging Face...")
            source = snapshot_download(
                repo_id=os.environ.get('MODEL_REPO_ID', MODEL_REPO_ID),
                revision=os.environ.get('MODEL_REVISION', MODEL_REVISION),
                token=token,
                allow_patterns=[
                    'config.json',
                    'preprocessor_config.json',
                    'model.safetensors',
                    'tagger_pipeline.py'
                ],
            )

        self.has_cuda = HAS_CUDA
        dev_label = f"GPU ({torch.cuda.get_device_name(0)}) FP16" if self.has_cuda else "CPU FP32"
        print(f"⚙️ Cargando pipeline en {dev_label}...")

        self.tagger = pipeline(
            model=str(source),
            image_processor=str(source),
            trust_remote_code=True,
            device=DEVICE_ARG,
            dtype=DTYPE_ARG,
            use_fast=False,
        )
        base_defaults = self.tagger.category_thresholds.copy()
        self.default_thresholds = {
            'general': base_defaults.get('general', 0.17),
            'character': base_defaults.get('character', 0.27),
            'clothing': 0.17,
            'pose': 0.17,
            'style': base_defaults.get('style', 0.15),
            'copyright': base_defaults.get('copyright', 0.24),
            'meta': base_defaults.get('meta', 0.17),
            'rating': base_defaults.get('rating', 0.41)
        }
        print(f"✅ Instalacion Completada!.")

    @torch.inference_mode()
    def __call__(self, image: Image.Image, thresholds: dict | None = None):
        if not isinstance(image, Image.Image):
            raise ValueError('Por favor proporciona una imagen válida.')
        if image.mode != 'RGB':
            image = image.convert('RGB')

        # Optimización de velocidad: si la imagen es gigante (>1536px), redimensionar preventivamente
        # para evitar cuellos de botella en memoria sin perder detalle en los tags
        max_dim = max(image.size)
        if max_dim > 1536:
            scale = 1536 / max_dim
            new_w = int(image.size[0] * scale)
            new_h = int(image.size[1] * scale)
            resample_mode = getattr(Image, 'Resampling', Image).BILINEAR
            image = image.resize((new_w, new_h), resample_mode)

        if thresholds is None:
            thresholds = self.default_thresholds.copy()

        # Umbral nativo mínimo para no descartar tags candidatos de ropa y pose
        internal_thresholds = {}
        for c in BASE_CATEGORIES:
            if c == 'general':
                internal_thresholds['general'] = min(
                    thresholds.get('general', 0.17),
                    thresholds.get('clothing', 0.17),
                    thresholds.get('pose', 0.17)
                )
            else:
                internal_thresholds[c] = thresholds.get(c, self.default_thresholds[c])

        raw_results = self.tagger(image, threshold=internal_thresholds)['results']

        clothing_dict = {}
        pose_dict = {}
        general_dict = {}

        for tag, score in raw_results.get('general', {}).items():
            if is_clothing_tag(tag):
                if score >= thresholds.get('clothing', 0.17):
                    clothing_dict[tag] = score
            elif is_pose_tag(tag):
                if score >= thresholds.get('pose', 0.17):
                    pose_dict[tag] = score
            else:
                if score >= thresholds.get('general', 0.17):
                    general_dict[tag] = score

        ordered = {
            'general': dict(sorted(general_dict.items(), key=lambda p: (-p[1], p[0]))),
            'character': dict(sorted(raw_results.get('character', {}).items(), key=lambda p: (-p[1], p[0]))),
            'clothing': dict(sorted(clothing_dict.items(), key=lambda p: (-p[1], p[0]))),
            'pose': dict(sorted(pose_dict.items(), key=lambda p: (-p[1], p[0]))),
            'style': dict(sorted(raw_results.get('style', {}).items(), key=lambda p: (-p[1], p[0]))),
            'copyright': dict(sorted(raw_results.get('copyright', {}).items(), key=lambda p: (-p[1], p[0]))),
            'meta': dict(sorted(raw_results.get('meta', {}).items(), key=lambda p: (-p[1], p[0]))),
            'rating': dict(sorted(raw_results.get('rating', {}).items(), key=lambda p: (-p[1], p[0])))
        }
        return ordered


def format_tags(scores, include_scores=False, keep_underscores=False):
    parts = []
    for name, score in scores.items():
        label = name if keep_underscores else name.replace('_', ' ')
        parts.append(f'{label} ({score:.2f})' if include_scores else label)
    return ', '.join(parts)


@lru_cache(maxsize=1)
def get_handler():
    return EndpointHandler()

# Carga anticipada del modelo
handler = get_handler()


### 🎨 Celda 2: Lanzar la Interfaz Web Gradio


In [ ]:
#@markdown # Iniciar
# ==============================================================================
# 2. INTERFAZ WEB GRADIO CON SLIDERS Y BOTONES PARA ROPA Y POSE
# ==============================================================================
import gradio as gr

def run_inference(image, include_scores, keep_underscores, combined_categories, *threshold_values):
    if image is None:
        raise gr.Error('Sube o pega una imagen primero.')

    thresholds = dict(zip(CATEGORIES, map(float, threshold_values)))
    try:
        active_handler = get_handler()
        started = time.perf_counter()
        results = active_handler(image, thresholds)
    except Exception as e:
        logging.exception('Error en inferencia')
        raise gr.Error(f'No se pudo procesar la imagen: {e}') from None

    elapsed = time.perf_counter() - started

    combined_parts = []
    for category in COMBINED_ORDER:
        if category in combined_categories and category in results:
            formatted = format_tags(results[category], include_scores, keep_underscores)
            if formatted:
                combined_parts.append(formatted)

    combined_str = ', '.join(combined_parts)

    outputs = [combined_str]
    outputs.extend(format_tags(results[c], include_scores, keep_underscores) for c in CATEGORIES)

    total = sum(len(tags) for tags in results.values())
    hw_str = f"GPU ({torch.cuda.get_device_name(0)})" if active_handler.has_cuda else "CPU FP32"
    outputs.extend([f'{total} tags detectados · {elapsed:.2f} s en {hw_str}', results])
    return outputs


theme = gr.themes.Soft(primary_hue='amber', secondary_hue='yellow', neutral_hue='stone').set(
    button_primary_background_fill='#e27b00',
    button_primary_background_fill_hover='#f59e0b',
    button_primary_text_color='#ffffff',
    button_primary_background_fill_dark='#e27b00',
    button_primary_background_fill_hover_dark='#f59e0b',
    button_primary_text_color_dark='#ffffff',
    slider_color='#e27b00',
    slider_color_dark='#e27b00',
)

CSS = '''
.gradio-container { width: 100% !important; min-width: 0 !important; max-width: 1120px !important; margin: auto; }
.gradio-container .tab-nav { flex-wrap: wrap !important; }

/* Estilo de etiquetas naranja tipo badge para sliders y componentes */
.badge-slider label > span,
.badge-label label > span,
.category-checkboxes label > span.wrap-inner {
    background-color: #e27b00 !important;
    color: #ffffff !important;
    font-weight: 700 !important;
    font-size: 0.88rem !important;
    padding: 4px 10px !important;
    border-radius: 6px !important;
    display: inline-block !important;
    margin-bottom: 6px !important;
}

/* Botones de selección de categoría tipo píldora */
.category-checkboxes .wrap {
    display: flex !important;
    flex-wrap: wrap !important;
    gap: 8px !important;
}
.category-checkboxes label {
    background-color: #27211c !important;
    border: 1px solid #4a3a2a !important;
    border-radius: 6px !important;
    padding: 6px 14px !important;
    cursor: pointer !important;
    transition: all 0.15s ease-in-out !important;
    display: inline-flex !important;
    align-items: center !important;
    margin: 0 !important;
}
.category-checkboxes label:hover {
    background-color: #3b2f23 !important;
    border-color: #e27b00 !important;
}
.category-checkboxes label:has(input:checked),
.category-checkboxes label.selected {
    background-color: #e27b00 !important;
    border-color: #e27b00 !important;
    color: #ffffff !important;
    font-weight: 700 !important;
    box-shadow: 0 2px 6px rgba(226, 123, 0, 0.4) !important;
}
.category-checkboxes label:has(input:checked) span,
.category-checkboxes label.selected span {
    color: #ffffff !important;
}

#banner img { width: 100%; height: auto; display: block; border-radius: 14px; max-height: 220px; object-fit: cover; }
#tag-button { min-height: 48px; font-weight: 700; font-size: 1.05rem; }
'''

def build_demo():
    h = get_handler()
    defaults = h.default_thresholds
    hw_info = f"🚀 Aceleración: GPU ({torch.cuda.get_device_name(0)}) FP16" if h.has_cuda else "⚙️ Modo: CPU"

    with gr.Blocks(title='PixAI Tagger v1.0 (Auto Colab)', analytics_enabled=False) as demo:
        banner_url = 'https://huggingface.co/spaces/pixai-labs/pixai-tagger-demo/resolve/main/static/banner.png'
        gr.HTML(f'<div id="banner"><img src="{banner_url}" alt="PixAI Tagger v1.0 — Anime multi-label tagger" onerror="this.style.display=\'none\'"></div>')
        gr.Markdown(f'### 🏷️ PixAI Tagger v1.0 — Anime Multi-label Tagger\n`{hw_info}` · Updated to PixAI Tagger v1.0. Upload an anime image to get tags for characters, clothing, styles, and more. 30,877 tags · May 2026 cutoff.')

        with gr.Row():
            with gr.Column(scale=1, min_width=300):
                image = gr.Image(label='Tu imagen', sources=['upload', 'clipboard'], type='pil', height=360)
                run = gr.Button('Etiquetar imagen', variant='primary', elem_id='tag-button')
                status = gr.Markdown('')
            with gr.Column(scale=1, min_width=300):
                combined = gr.Textbox(label='Combined tags (Etiquetas combinadas según categorías seleccionadas)', lines=12, interactive=False, buttons=['copy'])
                gr.Markdown('💡 *Copia directamente las etiquetas generadas para tu flujo de trabajo.*')

        with gr.Accordion('⚙️ Configuración de Umbrales y Categorías (Settings)', open=True):
            gr.Markdown('Ajusta los sliders de umbral (*thresholds*) para afinar la sensibilidad de cada categoría.')

            # Fila 1 de Sliders (General, Character, Style)
            with gr.Row():
                s_gen = gr.Slider(0, 1, value=defaults['general'], step=0.01, label='General threshold', elem_classes=['badge-slider'])
                s_char = gr.Slider(0, 1, value=defaults['character'], step=0.01, label='Character threshold', elem_classes=['badge-slider'])
                s_style = gr.Slider(0, 1, value=defaults['style'], step=0.01, label='Style threshold', elem_classes=['badge-slider'])

            # Fila 2 de Sliders (Clothing, Pose, Copyright / series)
            with gr.Row():
                s_cloth = gr.Slider(0, 1, value=defaults['clothing'], step=0.01, label='Clothing threshold', elem_classes=['badge-slider'])
                s_pose = gr.Slider(0, 1, value=defaults['pose'], step=0.01, label='Pose threshold', elem_classes=['badge-slider'])
                s_copy = gr.Slider(0, 1, value=defaults['copyright'], step=0.01, label='Copyright / series threshold', elem_classes=['badge-slider'])

            # Fila 3 de Sliders (Meta, Rating)
            with gr.Row():
                s_meta = gr.Slider(0, 1, value=defaults['meta'], step=0.01, label='Meta threshold', elem_classes=['badge-slider'])
                s_rating = gr.Slider(0, 1, value=defaults['rating'], step=0.01, label='Rating threshold', elem_classes=['badge-slider'])

            sliders = [s_gen, s_char, s_cloth, s_pose, s_style, s_copy, s_meta, s_rating]

            with gr.Row():
                include_scores = gr.Checkbox(value=False, label='Include confidence scores')
                keep_underscores = gr.Checkbox(value=False, label='Keep underscores in tag names')

            # Botones estilo píldora para seleccionar categorías
            combined_categories = gr.CheckboxGroup(
                choices=[
                    ('General', 'general'),
                    ('Character', 'character'),
                    ('Clothing', 'clothing'),
                    ('Pose', 'pose'),
                    ('Style', 'style'),
                    ('Copyright / series', 'copyright'),
                    ('Meta', 'meta'),
                    ('Rating', 'rating')
                ],
                value=list(CATEGORIES),
                label='Categories in combined tags',
                elem_classes=['category-checkboxes', 'badge-label']
            )

        gr.Markdown('### 📂 Etiquetas desglosadas por categoría')
        category_outputs = []
        with gr.Tabs():
            for category in CATEGORIES:
                icon = '👗 ' if category == 'clothing' else ('🧘 ' if category == 'pose' else '')
                with gr.Tab(f'{icon}{LABELS[category]}'):
                    category_outputs.append(gr.Textbox(label=f'Tags de {LABELS[category]}', lines=7, interactive=False, buttons=['copy']))

        with gr.Accordion('🔍 Puntuaciones brutas (JSON)', open=False):
            raw = gr.JSON(label='Tags y probabilidades completas')

        clear = gr.ClearButton([image, combined, *category_outputs, status, raw], value='Limpiar')
        gr.Markdown('[PixAI Tagger v1.0 en Hugging Face](https://huggingface.co/pixai-labs/pixai-tagger-v1.0) · [Space original](https://huggingface.co/spaces/pixai-labs/pixai-tagger-demo)')

        run.click(
            run_inference,
            inputs=[image, include_scores, keep_underscores, combined_categories, *sliders],
            outputs=[combined, *category_outputs, status, raw],
            api_name='tag',
            concurrency_limit=1
        )
    return demo

# Iniciar la aplicación Gradio con enlace público para Colab
demo = build_demo()
demo.queue(max_size=16).launch(share=True, theme=theme, css=CSS, show_error=True)
